In [1]:
import os
import torch
from dotenv import load_dotenv

load_dotenv()
access_token = os.environ.get("HF_TOKEN")
if access_token is None:
    raise ValueError("Set HF_TOKEN in your environment or in a local .env file.")

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='cuda')

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b")

test_config = {
    "tokenizer": tokenizer,
    "batch_size": 8,
    "context_length": 512,
    "vocab_size": len(tokenizer),
    "embedding_dim": 512,
    "n_layers": 8,
    "n_heads": 8,
    "n_kv_heads": 4,
}

batch_size = test_config["batch_size"]
context_length = test_config["context_length"]
vocab_size = test_config["vocab_size"]
embedding_dim = test_config["embedding_dim"]
n_layers = test_config["n_layers"]
n_heads = test_config["n_heads"]
n_kv_heads = test_config["n_kv_heads"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from data import create_python_dataset, CoderDataset
from torch.utils.data import DataLoader

python_raw = create_python_dataset(access_token)

val_raw = python_raw.take(200)
train_raw = python_raw.skip(200).take(5000)

train_set = CoderDataset(train_raw, tokenizer, context_length)
val_set = CoderDataset(val_raw, tokenizer, context_length)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=0)

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

In [4]:
def generate_tokens(input, num_tokens, model):
  encoded = tokenizer.encode(input, return_tensors="pt").to(device)


  model.eval()
  with torch.no_grad():
    for _ in range(num_tokens):
      model_input = encoded[:, -context_length:]
      output = model(model_input)
      next_token_pred = torch.argmax(output[:, -1, :], dim=-1).unsqueeze(1)
      encoded = torch.cat((encoded, next_token_pred), dim=1)
  return encoded


In [5]:
def evaluate_model(data_loader, model, loss_fn):
  model.eval()
  losses = []
  with torch.no_grad():
    for x, y in iter(data_loader):
      x, y = x.to(device), y.to(device)
      logits = model(x)
      loss = loss_fn(torch.flatten(logits, end_dim=1), torch.flatten(y))
      losses.append(loss.item())

  avg_loss = sum(losses) / len(losses)
  return avg_loss


In [6]:
def train_model(train_loader, val_loader, model, optimizer, loss_fn, max_steps=1000):
    model.train()
    train_losses = []
    val_losses = []
    i = 0

    for x, y in iter(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        logits = model(x)
        loss = loss_fn(torch.flatten(logits, end_dim=1), torch.flatten(y))
        loss.backward()
        train_losses.append(loss.item())
        optimizer.step()

        i += 1
        if i % 100 == 0:
            val_loss = evaluate_model(val_loader, model, loss_fn)
            val_losses.append(val_loss)
            print(
                f"Step {i}: "
                f"Train loss = {loss.item()} "
                f"Val loss = {val_loss}"
            )
            model.train()
        if i >= max_steps:
            break
    return train_losses, val_losses

In [7]:
from model import LLM
from torch import optim, nn

model = LLM(
    vocab_size,
    context_length,
    embedding_dim,
    n_layers,
    n_heads,
    n_kv_heads,
    intermediate_dim=embedding_dim * 4
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

In [8]:
train_losses, val_losses = train_model(
      train_loader,
      val_loader,
      model,
      optimizer,
      loss_fn,
      max_steps=1000
)


decoded = tokenizer.decode(generate_tokens("def hello_world", 15, model)[0])
print(decoded)

Step 100: Train loss = 32.50481414794922 Val loss = 30.46359157562256
Step 200: Train loss = 20.878311157226562 Val loss = 20.252254561374063
Step 300: Train loss = 14.056575775146484 Val loss = 15.575902252866511
Step 400: Train loss = 11.021220207214355 Val loss = 13.211060047149658
Step 500: Train loss = 12.001910209655762 Val loss = 11.754403942509702
Step 600: Train loss = 11.022907257080078 Val loss = 10.730024136995015
Step 700: Train loss = 10.62459945678711 Val loss = 9.856098049565366
Step 800: Train loss = 10.278763771057129 Val loss = 9.654924045529281
Step 900: Train loss = 7.745637893676758 Val loss = 8.75070878079063
Step 1000: Train loss = 8.674416542053223 Val loss = 8.530949220322727
def hello_world_dict_dict_dict_dict_dict_dict_dict_
